# Meridiana — Test Suite su Google Colab

Questo notebook esegue i test unitari del layer DB di Meridiana (mock-based, nessun DB reale richiesto).  
Utile per verificare gli avanzamenti su qualsiasi PC senza installare nulla localmente.

**Cosa viene testato:** logica `db/` (comuni, partite, possessori, ricerca, audit, variazioni, immobili, backup, documenti)  
**Cosa NON viene testato:** GUI PyQt6 (richiede display fisico)

---

## 1. Clone del repository

In [ ]:
import os

REPO_URL = "https://github.com/santoromarco74/catasto.git"
REPO_DIR = "catasto"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print("Repo già presente, aggiorno...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!git log --oneline -5

## 2. Installazione dipendenze (solo non-GUI)

In [ ]:
!pip install -q \
    psycopg2-binary \
    pandas \
    openpyxl \
    fpdf2 \
    bcrypt \
    keyring \
    odfpy \
    markdown \
    pytest \
    pytest-cov \
    pytest-mock

print("Dipendenze installate.")

## 3. Configurazione ambiente CI

In [ ]:
import os

# Modalità CI: disabilita prompt interattivi e GUI
os.environ["CI"] = "true"
os.environ["QT_QPA_PLATFORM"] = "offscreen"

# Credenziali DB fittizie (non usate nei test unitari con mock)
os.environ["DB_HOST"] = "localhost"
os.environ["DB_USER"] = "postgres"
os.environ["DB_PASS"] = "postgres"
os.environ["DB_NAME"] = "catasto_storico"
os.environ["DB_PORT"] = "5432"

print("Ambiente configurato per CI.")

## 4. Esecuzione test unitari (DB layer — mock)

In [ ]:
!python -m pytest tests/unit/ -v --tb=short \
    --ignore=tests/unit/test_new_modules.py \
    -p no:cacheprovider \
    2>&1 | head -120

## 5. Coverage report

In [ ]:
!python -m pytest tests/unit/ -v \
    --ignore=tests/unit/test_new_modules.py \
    --cov=db --cov=catasto_db_manager --cov=catasto_exceptions \
    --cov-report=term-missing \
    -p no:cacheprovider \
    -q 2>&1

## 6. Verifica import moduli principali

In [ ]:
moduli = [
    "config",
    "app_paths",
    "app_utils",
    "catasto_exceptions",
    "catasto_db_manager",
    "db.base",
    "db.comuni",
    "db.partite",
    "db.possessori",
    "db.localita",
    "db.immobili",
    "db.variazioni",
    "db.ricerca",
    "db.audit",
    "db.utenti",
    "db.backup",
    "db.documenti",
    "db.stats",
    "db.io",
]

import importlib
ok, fail = [], []
for m in moduli:
    try:
        importlib.import_module(m)
        ok.append(m)
    except Exception as e:
        fail.append((m, str(e)))

print(f"\n✓ {len(ok)} moduli importati correttamente")
if fail:
    print(f"✗ {len(fail)} errori:")
    for m, e in fail:
        print(f"  - {m}: {e}")
else:
    print("Nessun errore di import.")

## 7. Info build

Versione corrente e ultimi commit.

In [ ]:
import sys
sys.path.insert(0, '.')
from config import APP_VERSION
print(f"Meridiana v{APP_VERSION}")
print()
!git log --oneline -10